In [ ]:
import pandas as pd
import numpy as np
import time
import os
import shutil

n_sales = 1000000

sales_df = pd.DataFrame({
    'date': pd.date_range('2023-01-01', periods=n_sales, freq='min'), 
    'region': np.random.choice(['North', 'South', 'East', 'West'], n_sales),
    'category': np.random.choice(['Electronics', 'Clothing', 'Home', 'Toys'], n_sales),
    'sales_amount': np.random.uniform(10, 500, n_sales)
})


sales_df['year'] = sales_df['date'].dt.year
sales_df['month'] = sales_df['date'].dt.month

for folder in ['partitioned_by_date', 'partitioned_by_region', 'partitioned_by_category']:
    if os.path.exists(folder): shutil.rmtree(folder)

sales_df.to_parquet('partitioned_by_date', partition_cols=['year', 'month'])
print("Divided by date (Year/Month).")

sales_df.to_parquet('partitioned_by_region', partition_cols=['region'])
print("The division was by region(Region).")

sales_df.to_parquet('partitioned_by_category', partition_cols=['category'])
print("The division was by category (Category).")


start = time.time()
res_none = sales_df[sales_df['region'] == 'North']
time_no_partition = time.time() - start


start = time.time()

df_partitioned = pd.read_parquet('partitioned_by_region/region=North')
time_with_partition = time.time() - start


print(f"Results:")
print(f"query time without division: {time_no_partition:.4f} s")
print(f"Query time using partitioning: {time_with_partition:.4f} s")
print(f"Speed improvement percentage: {time_no_partition / time_with_partition:.2f}x faster")


folder_size = sum(os.path.getsize(os.path.join('partitioned_by_region/region=North', f)) 
                 for f in os.listdir('partitioned_by_region/region=North')) / 1024
print(f"Northern Region Section Size (North Partition): {folder_size:.2f} KB")